# hyperspectral CLEM: WL vs FRL Comparison

Analyses low-temperature hyperspectral fluorescence imaging data from *Chroococcidiopsis thermalis* cells grown under white light (WL) or far-red light (FRL) conditions.

## Pipeline overview
1. Load per-cell emission spectra (440 nm and 560 nm excitation) from CSV files
2. Filter cells by minimum peak intensity and peak wavelength
3. Interpolate to a common 1 nm wavelength grid (no extrapolation)
4. Normalise to the chlorophyll emission maximum (710–780 nm)
5. Plot WL vs FRL mean ± SD spectra with emission peak labels
6. Plot individual cell spectrum for CLEM correlation

## Distance/intensity definitions
- **Peak intensity**: maximum intensity over the full measured wavelength range per cell
- **Normalisation**: each cell's spectrum is divided by its maximum intensity within 710–780 nm (Chl *a* emission window); cells are excluded if this value is zero or below the minimum intensity threshold
- **Interpolation**: linear interpolation to a 1 nm grid within each cell's measured wavelength span; no extrapolation beyond measured endpoints

## Assumptions and limitations
- Cells with peak intensity below `MIN_INTENSITY` or peak emission below `MIN_PEAK_WL_*` are excluded as likely non-photosynthetic or out-of-focus
- Datasets collected under different acquisition parameters are pooled after interpolation; the 1 nm grid normalises differences in spectral sampling
- Peak detection uses a plateau-aware prominence filter; parameters may need adjustment for different spectral shapes

**Dependencies:** numpy, pandas, matplotlib, scipy

**Citation:** If used, please cite this repository [DOI placeholder].

## 1. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## 2. User Parameters

In [ ]:
# ── Filtering ─────────────────────────────────────────────────────────────────

MIN_INTENSITY      = 5.0    # minimum peak intensity; cells below this are excluded
MIN_PEAK_WL_440    = 680.0  # minimum wavelength (nm) of peak for 440 nm excitation data
MIN_PEAK_WL_560    = 615.0  # minimum wavelength (nm) of peak for 560 nm excitation data

# ── Normalisation ─────────────────────────────────────────────────────────────

NORM_RANGE = (710, 780)     # wavelength range (nm) for max-intensity normalisation

# ── Plot output range ─────────────────────────────────────────────────────────

PLOT_WL_RANGE = (600, 800)  # wavelength window (nm) for WL vs FRL comparison plots
TOP_K_PEAKS   = 5           # max number of emission peaks to label per spectrum

# ── Colors ────────────────────────────────────────────────────────────────────

CAT_COLOR = {"WL": "#3986a1ff", "FRL": "#a41858ff"}

# ── Output directory ──────────────────────────────────────────────────────────

OUTDIR = Path("hyperspectral_output")

# ── Single-cell CLEM plot ─────────────────────────────────────────────────────
# Image IDs to plot individually (from datasets_440_norm after normalisation)

SINGLE_CELL_IDS = [
    "cell 1",
    "cell 2",
]

## 3. Data Paths

In [ ]:
dataset_paths_440 = {
    "FRL": [
        "/path/to/combined_intensity_data.csv",
    ],
    "WL": [
        "/path/to/combined_intensity_data.csv",
    ],
}

dataset_paths_560 = {
    "FRL": [
        "/path/to/combined_intensity_data.csv",
    ],
    "WL": [
       "/path/to/combined_intensity_data.csv",
    ],
}

## 4. Helper Functions

Run once.

In [ ]:
def load_datasets(paths_dict):
    """Load CSVs, parse image_ID and cell_ID from the image column."""
    datasets = {}
    for cat, paths in paths_dict.items():
        datasets[cat] = {}
        for path in paths:
            df = pd.read_csv(path)
            if "image" not in df.columns:
                print(f"[warn] 'image' column missing: {path}")
                continue
            df["image_ID"] = df["image"].str.split('_MIP_seg.npy__mean_intensity_').str[0]
            df["cell_ID"]  = df["image"].str.extract(r'_MIP_seg.npy__mean_intensity_(\d+)$')[0]
            datasets[cat][Path(path).parent.as_posix()] = df
    return datasets


def filter_cells(datasets, min_intensity, min_peak_wl):
    """Remove cells below a minimum peak intensity or with peak emission below min_peak_wl."""
    filtered = {}
    stats = []
    for cat, dct in datasets.items():
        filtered[cat] = {}
        for name, df in dct.items():
            df2 = df.copy()
            df2["wavelength"] = pd.to_numeric(df2["wavelength"], errors="coerce")
            df2["intensity"]  = pd.to_numeric(df2["intensity"],  errors="coerce")
            df2 = df2.dropna(subset=["wavelength", "intensity", "image"])
            idxmax = df2.groupby("image")["intensity"].idxmax()
            peaks  = (df2.loc[idxmax, ["image","wavelength","intensity"]]
                       .rename(columns={"wavelength":"peak_wl","intensity":"peak_int"})
                       .set_index("image"))
            keep = peaks.index[
                (peaks["peak_int"] >= min_intensity) &
                (peaks["peak_wl"]  >= min_peak_wl)
            ]
            filtered[cat][name] = df2[df2["image"].isin(keep)].copy()
            stats.append({"category": cat, "dataset": Path(name).name,
                          "before": df2["image"].nunique(), "after": len(keep)})
    return filtered, pd.DataFrame(stats)


def interpolate_1nm(datasets, grid=None):
    """Interpolate each cell spectrum to a common 1 nm grid. No extrapolation."""
    if grid is None:
        mins, maxs = [], []
        for dct in datasets.values():
            for df in dct.values():
                w = pd.to_numeric(df["wavelength"], errors="coerce").dropna()
                if not w.empty:
                    mins.append(w.min()); maxs.append(w.max())
        grid = np.arange(int(np.floor(min(mins))), int(np.ceil(max(maxs))) + 1, dtype=int)

    interp = {}
    for cat, dct in datasets.items():
        interp[cat] = {}
        for name, df in dct.items():
            tmp = df[["image","wavelength","intensity"]].copy()
            tmp["wavelength"] = pd.to_numeric(tmp["wavelength"], errors="coerce")
            tmp["intensity"]  = pd.to_numeric(tmp["intensity"],  errors="coerce")
            tmp = tmp.dropna().sort_values(["image","wavelength"])
            rows = []
            for img, g in tmp.groupby("image"):
                g = g.groupby("wavelength", as_index=False)["intensity"].mean()
                x, y = g["wavelength"].to_numpy(), g["intensity"].to_numpy()
                inside = (grid >= x.min()) & (grid <= x.max())
                if not inside.any(): continue
                rows.append(pd.DataFrame({"image": img,
                                          "wavelength": grid[inside],
                                          "intensity": np.interp(grid[inside], x, y)}))
            interp[cat][name] = pd.concat(rows, ignore_index=True) if rows else \
                pd.DataFrame(columns=["image","wavelength","intensity"])
    return interp, grid


def normalise(datasets, norm_range=(710, 780)):
    """Divide each cell spectrum by its maximum intensity within norm_range."""
    lo, hi = norm_range
    normed = {}
    for cat, dct in datasets.items():
        normed[cat] = {}
        for name, df in dct.items():
            df2   = df.copy()
            denom = (df2.loc[df2["wavelength"].between(lo, hi)]
                       .groupby("image")["intensity"].max())
            df2["intensity"] = df2["intensity"] / df2["image"].map(denom)
            normed[cat][name] = df2
    return normed


def aggregate_mean_sd(datasets, cat, wl_range=None):
    """Compute mean and SD across all cells in a category."""
    frames = [df[["wavelength","intensity"]] for df in datasets[cat].values()
              if {"wavelength","intensity"}.issubset(df.columns)]
    if not frames: return None
    all_cat = pd.concat(frames, ignore_index=True)
    if wl_range:
        lo, hi = wl_range
        all_cat = all_cat[(all_cat["wavelength"] >= lo) & (all_cat["wavelength"] <= hi)]
    return (all_cat.groupby("wavelength", as_index=False)["intensity"]
                   .agg(mean="mean", std="std")
                   .sort_values("wavelength"))


def find_peaks_robust(x, y, half_width=7, prom_window=10, min_distance_nm=5,
                      top_k=5, relative_prominence=0.001, smooth_win=2):
    """Plateau-aware prominence-filtered local maximum finder."""
    x, y = np.asarray(x), np.asarray(y, dtype=float)
    if smooth_win and smooth_win > 1:
        w = int(smooth_win) | 1
        y = np.convolve(y, np.ones(w)/w, mode="same")
    ok = ~np.isnan(y)
    x, y = x[ok], y[ok]
    n = len(y)
    if n == 0: return np.array([]), np.array([])

    eps = 1e-4
    cand = [i for i in range(half_width, n - half_width)
            if (y[i] >= np.nanmax(y[i-half_width:i]) - eps and
                y[i] >= np.nanmax(y[i+1:i+1+half_width]) - eps and
                (y[i] > np.nanmax(y[i-half_width:i]) + eps or
                 y[i] > np.nanmax(y[i+1:i+1+half_width]) + eps))]
    if not cand: return np.array([]), np.array([])

    idx = np.array(cand, dtype=int)
    jumps = np.where(np.diff(idx) > 1)[0]
    peaks = np.array([(idx[s] + idx[e]) // 2
                      for s, e in zip(np.r_[0, jumps+1], np.r_[jumps, len(idx)-1])], dtype=int)

    dyn = max(1e-12, np.nanmax(y) - np.nanmin(y))
    prom = []
    for p in peaks:
        l0, r0 = max(0, p-prom_window), min(n, p+prom_window+1)
        prom.append(y[p] - max(np.min(y[l0:p+1]), np.min(y[p:r0])))
    prom = np.array(prom)
    keep = prom >= relative_prominence * dyn
    peaks, prom = peaks[keep], prom[keep]
    if peaks.size == 0: return np.array([]), np.array([])

    order = np.argsort(prom)[::-1]
    peaks = peaks[order]
    selected = []
    for p in peaks:
        if not selected or np.min(np.abs(x[p] - x[selected])) >= min_distance_nm:
            selected.append(p)
        if len(selected) >= top_k: break
    sel = np.array(selected, dtype=int)
    return x[sel], y[sel]

## 5. Load, Filter, Interpolate, Normalise

In [ ]:
OUTDIR.mkdir(parents=True, exist_ok=True)

# Load
datasets_440 = load_datasets(dataset_paths_440)
datasets_560 = load_datasets(dataset_paths_560)

# Filter
datasets_440_filt, stats_440 = filter_cells(datasets_440, MIN_INTENSITY, MIN_PEAK_WL_440)
datasets_560_filt, stats_560 = filter_cells(datasets_560, MIN_INTENSITY, MIN_PEAK_WL_560)

print("440 nm — cells before/after filtering:")
display(stats_440)
print("\n560 nm — cells before/after filtering:")
display(stats_560)

# Interpolate
datasets_440_interp, grid_440 = interpolate_1nm(datasets_440_filt)
datasets_560_interp, grid_560 = interpolate_1nm(datasets_560_filt)

# Normalise
datasets_440_norm = normalise(datasets_440_interp, NORM_RANGE)
datasets_560_norm = normalise(datasets_560_interp, NORM_RANGE)

# Report cell counts per condition
for label, ds in [("440 nm", datasets_440_norm), ("560 nm", datasets_560_norm)]:
    print(f"\n{label} cell counts:")
    for cat, dct in ds.items():
        n = sum(df["image"].nunique() for df in dct.values())
        print(f"  {cat}: {n} cells")

## 6. WL vs FRL Mean ± SD Spectra

Per-cell traces shown at low opacity; bold line = mean across all cells; shaded band = ±1 SD. Emission peaks on the mean curve are labelled with their wavelength.

In [ ]:
def plot_wl_frl(datasets_norm, excitation_nm, wl_range=PLOT_WL_RANGE,
                top_k=TOP_K_PEAKS, savepath=None):
    """Two-column grid per condition: left = individual traces, right = mean ± SD."""
    conditions = [cat for cat in ("WL", "FRL") if cat in datasets_norm]
    n = len(conditions)
    lo, hi = wl_range

    fig, axes = plt.subplots(n, 2, figsize=(8, 3.5 * n),
                             sharex=True, sharey="row",
                             gridspec_kw={"wspace": 0.08, "hspace": 0.35})
    axes = np.atleast_2d(axes)

    for row, cat in enumerate(conditions):
        color = CAT_COLOR[cat]
        agg   = aggregate_mean_sd(datasets_norm, cat, wl_range=wl_range)
        n_cells = sum(df["image"].nunique() for df in datasets_norm[cat].values())

        # ── Left: individual traces ──────────────────────────────────────────
        ax_traces = axes[row, 0]
        for df in datasets_norm[cat].values():
            df_clip = df[(df["wavelength"] >= lo) & (df["wavelength"] <= hi)].dropna(subset=["intensity"])
            for _, g in df_clip.groupby("image"):
                ax_traces.plot(g["wavelength"], g["intensity"],
                               alpha=0.05, linewidth=0.6, color=color)
        ax_traces.set_title(f"{cat}  |  individual cells (n={n_cells})", fontsize=9)
        ax_traces.set_ylabel("Normalized intensity")
        if row == n - 1:
            ax_traces.set_xlabel("Wavelength (nm)")

        # ── Right: mean ± SD ─────────────────────────────────────────────────
        ax_mean = axes[row, 1]
        if agg is not None and not agg.empty:
            x = agg["wavelength"].to_numpy()
            m = agg["mean"].to_numpy()
            s = np.nan_to_num(agg["std"].to_numpy(), nan=0.0)
            ax_mean.fill_between(x, m - s, m + s, color=color, alpha=0.25, linewidth=0)
            ax_mean.plot(x, m, color=color, linewidth=2.0)

            px, py = find_peaks_robust(x, m, top_k=top_k)
            if px.size:
                ax_mean.scatter(px, py, s=28, color=color, zorder=4)
                for xx, yy in zip(px, py):
                    ax_mean.text(xx, yy, f"{int(round(xx))}",
                                 color=color, fontsize=9, ha="center", va="bottom",
                                 bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                           alpha=0.8, ec="none"), zorder=5)
        ax_mean.set_title(f"{cat}  |  mean ± SD", fontsize=9)
        if row == n - 1:
            ax_mean.set_xlabel("Wavelength (nm)")

    fig.suptitle(f"{excitation_nm} nm excitation", fontsize=11, y=1.01)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
        print(f"Saved: {savepath}")
    plt.show()


plot_wl_frl(datasets_440_norm, excitation_nm=440,
            savepath=OUTDIR / "WLvsFRL_440.svg")

plot_wl_frl(datasets_560_norm, excitation_nm=560,
            savepath=OUTDIR / "WLvsFRL_560.svg")

## 7. Single-Cell Spectra for CLEM

Individual cell spectra identified by their image ID, retrieved from the normalised 440 nm dataset.

In [ ]:
def plot_single_cell(cell_id, datasets_norm, outdir):
    """Find and plot a single cell spectrum from the normalised dataset."""
    hits = [(cond, path)
            for cond, dct in datasets_norm.items()
            for path, df in dct.items()
            if "image" in df.columns and (df["image"] == cell_id).any()]

    if not hits:
        print(f"[warn] Cell ID not found: {cell_id}")
        return
    if len(hits) > 1:
        print(f"[warn] Cell ID found in multiple datasets:")
        for cond, path in hits: print(f"  {cond} | {path}")

    cond, path = hits[0]
    df_cell = (datasets_norm[cond][path]
               .loc[lambda d: d["image"] == cell_id, ["wavelength", "intensity"]]
               .sort_values("wavelength"))

    x, y = df_cell["wavelength"].to_numpy(), df_cell["intensity"].to_numpy()
    peak_idx = y.argmax()
    peak_x, peak_y = x[peak_idx], y[peak_idx]

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot(x, y, linewidth=2, color=CAT_COLOR.get(cond, "black"))
    ax.scatter([peak_x], [peak_y], zorder=3, color=CAT_COLOR.get(cond, "black"))
    ax.annotate(f"{int(round(peak_x))} nm",
                xy=(peak_x, peak_y), xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=9,
                arrowprops=dict(arrowstyle="-", lw=0.8))
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Normalized intensity")
    ax.set_title(f"{cond}  |  {Path(path).name}\ncell {cell_id.split('_')[-1]}",
                 fontsize=8)
    plt.tight_layout()

    safe_id = cell_id.replace(" ", "_").replace("/", "-")
    out = outdir / f"{safe_id}_spectrum.svg"
    plt.savefig(out, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.show()
    print(f"Peak: {peak_x} nm  |  intensity: {peak_y:.4f}")


for cell_id in SINGLE_CELL_IDS:
    plot_single_cell(cell_id, datasets_440_norm, OUTDIR)